In [1]:
import torch


class AttnOrderEval:

    def _build_geometry(self):
        T, L = self.T, self.L
        dev = self.attn.device

        step_of = torch.full((L,), -1, dtype=torch.long, device=dev)   # [L] step at which a position is unmasked, -1 = never
        step_of[self.order] = torch.arange(T, device=dev)

        t_idx = torch.arange(T, device=dev).view(T, 1)
        future_gap = step_of.view(1, L) - t_idx                        # [T, L]  gap>0 -> future candidate, gap=1 -> next
        cand_mask = future_gap > 0                                     # [T, L]  still-masked at step t

        neg_inf = torch.finfo(self.attn.dtype).min
        S = self.attn.masked_fill(~cand_mask, neg_inf)                 # [T, L]  attention restricted to future candidates

        n_cand = (T - 1) - torch.arange(T, device=dev)                 # [T]  number of still-masked tokens after step t
        return future_gap, cand_mask, S, n_cand
    # end

    def __init__(self, attn_rows, order):                             # attn_rows [T, L], order [T] long
        assert attn_rows.dim() == 2, "attn_rows.dim(): {} == 2 false".format(attn_rows.dim())
        assert order.dim() == 1, "order.dim(): {} == 1 false".format(order.dim())

        self.attn = attn_rows.to(torch.float64)
        self.order = order.to(torch.long)

        self.T, self.L = self.attn.shape

        self.future_gap, self.cand_mask, self.S, self.n_cand = self._build_geometry()
    # end

    def get_attn(self):
        return self.attn
    # end

    def get_order(self):
        return self.order
    # end

    def get_future_gap(self):
        return self.future_gap
    # end

    def get_S(self):
        return self.S
    # end

    def get_n_cand(self):
        return self.n_cand
    # end

    def _nan_invalid(self, value, valid):                            # [T], [T] bool -> [T] with nan where invalid
        nan = torch.full_like(value, float("nan"))
        return torch.where(valid, value, nan)
    # end

    def recall_at_h(self, h):   # of the next-h soonest tokens, how many land in q's top-h attended; per-step [T], use .nanmean()
        rel = (self.future_gap >= 1) & (self.future_gap <= h)        # [T, L]  soonest-h == relevant
        toph = self.S.topk(h, dim=-1).indices                       # [T, h]  predicted top-h candidates
        hit = rel.gather(-1, toph).double().sum(-1)                 # [T]
        recall = hit / h                                            # [T]

        valid = self.n_cand >= h                                    # need h relevant tokens to exist
        return self._nan_invalid(recall, valid)
    # end

    def pr_auc(self, h):   # average precision, positives = next-h soonest, scored by attention; per-step [T], use .nanmean()
        idx_sorted = self.S.argsort(dim=-1, descending=True)        # [T, L]  non-candidates (-inf) sink to the end
        gap_sorted = self.future_gap.gather(-1, idx_sorted)         # [T, L]

        is_cand = gap_sorted > 0                                    # [T, L]  a retrieved candidate at this rank
        is_pos = (gap_sorted >= 1) & (gap_sorted <= h)              # [T, L]  a soonest-h positive

        tp = is_pos.cumsum(-1).double()                            # [T, L]
        retrieved = is_cand.cumsum(-1).double().clamp_min(1.0)     # [T, L]  rank among candidates only
        precision = tp / retrieved                                # [T, L]

        P = is_pos.double().sum(-1)                                # [T]  == min(h, n_cand)
        ap = (precision * is_pos.double()).sum(-1) / P.clamp_min(1.0)   # [T]

        valid = self.n_cand > h                                    # need >=1 positive and >=1 negative
        return self._nan_invalid(ap, valid)
    # end

    def ndcg_at_h(self, H, gain="linear"):   # graded by soonness, attention ranks the candidates; per-step [T], use .nanmean()
        dev = self.attn.device

        base = (H - (self.future_gap - 1)).clamp(min=0).double()    # [T, L]  gap=1 -> H, gap=H -> 1, gap>H -> 0
        if gain == "exp":
            base = torch.exp2(base) - 1.0
        # end
        grade = base * self.cand_mask.double()                     # [T, L]  zero out non-candidates

        disc = 1.0 / torch.log2(torch.arange(H, device=dev).double() + 2.0)   # [H]
        pred = self.S.topk(H, dim=-1).indices                      # [T, H]  predicted ranking
        dcg = (grade.gather(-1, pred) * disc).sum(-1)              # [T]

        ideal = grade.topk(H, dim=-1).values                       # [T, H]  ideal ranking
        idcg = (ideal * disc).sum(-1)                              # [T]

        ndcg = dcg / idcg.clamp_min(torch.finfo(torch.float64).tiny)   # [T]
        valid = (self.n_cand >= 2) & (idcg > 0)
        return self._nan_invalid(ndcg, valid)
    # end



In [2]:

if __name__ == "__main__":
    torch.manual_seed(0)
    T = L = 64
    order = torch.randperm(L)

    step_of = torch.full((L,), -1, dtype=torch.long)
    step_of[order] = torch.arange(T)
    
    gap = step_of.view(1, L) - torch.arange(T).view(T, 1)          # [T, L]

    attn_perfect = torch.where(gap > 0, 1.0 / gap.clamp_min(1).double(), torch.zeros(1, dtype=torch.float64))
    attn_random = torch.rand(T, L, dtype=torch.float64)

    for name, attn in [("perfect", attn_perfect), ("random ", attn_random)]:
        ev = AttnOrderEval(attn, order)
        r = ev.recall_at_h(5).nanmean().item()
        p = ev.pr_auc(5).nanmean().item()
        nd = ev.ndcg_at_h(10).nanmean().item()
        print("{}  recall@5={:.3f}  pr_auc@5={:.3f}  ndcg@10={:.3f}".format(name, r, p, nd))
    # end

perfect  recall@5=1.000  pr_auc@5=1.000  ndcg@10=1.000
random   recall@5=0.186  pr_auc@5=0.280  ndcg@10=0.370


In [3]:
def generate_masked_attn(a, b, value_mask = -1):
    a = a.detach().clone()

    for i in range(b.size()[0]):
        a[i, b[:i]] = -1
    # end
    return a
# end

if __name__ == "__main__":
    torch.manual_seed(0)
    T = L = 64

    unmask = torch.load('stats/0/unmask_96_160.pt')
    unmask = unmask.squeeze(-1) - 32
    order = unmask

    step_of = torch.full((L,), -1, dtype=torch.long)
    step_of[order] = torch.arange(T)
    
    gap = step_of.view(1, L) - torch.arange(T).view(T, 1)          # [T, L]

    attn_perfect = torch.where(gap > 0, 1.0 / gap.clamp_min(1).double(), torch.zeros(1, dtype=torch.float64))
    attn_random = torch.rand(T, L, dtype=torch.float64)

    attn = torch.load('stats/0/attn_96_160.pt')
    attn_last = attn[:,-1,:,:].squeeze(1) # -> [step, attn]
    attn_last_switched = attn_last[unmask, :]

    for name, attn in [("perfect", attn_perfect), ("random ", attn_random), ("attn", attn_last_switched)]:
        ev = AttnOrderEval(attn, order)
        r = ev.recall_at_h(5).nanmean().item()
        p = ev.pr_auc(5).nanmean().item()
        nd = ev.ndcg_at_h(10).nanmean().item()
        print("{}  recall@5={:.3f}  pr_auc@5={:.3f}  ndcg@10={:.3f}".format(name, r, p, nd))
    # end

IndexError: index 65 is out of bounds for dimension 0 with size 64